<a href="https://colab.research.google.com/github/sathvikeppakayala/predictive_analytics/blob/house_price_prediction/CHurnPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
data = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [60]:
data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [61]:
data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [62]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [63]:
data.drop('customerID', inplace=True, axis=1, errors='coerce')
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data.dropna(inplace=True)
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

In [64]:
categorical_cols = data.select_dtypes(include='object').columns
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

In [65]:
x = data.drop('Churn', axis=1, errors='coerce')
y = data['Churn']
y.head()

,Churn
0,0
1,0
2,1
3,0
4,1


In [66]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [71]:
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.3, random_state=42)
lr = LogisticRegression(max_iter=1000)
lr.fit(x_train, y_train)
lr_preds = lr.predict(x_test)
knn_params = {'n_neighbors': list(range(3, 15))}
from sklearn.model_selection import GridSearchCV
knn = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5)
knn.fit(x_train, y_train)
knn_preds = knn.predict(x_test)
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
rf = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=5, cv=5)
rf.fit(x_train, y_train)
rf_preds = rf.predict(x_test)
gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.1, 0.05],
    'max_depth': [3, 5]
}
gb = GridSearchCV(GradientBoostingClassifier(random_state=42), gb_params, cv=5)
gb.fit(x_train, y_train)
gb_preds = gb.predict(x_test)
from sklearn.ensemble import VotingClassifier
ensemble = VotingClassifier(estimators=[
    ('lr', lr),
    ('knn', knn.best_estimator_),
    ('rf', rf.best_estimator_),
    ('gb', gb.best_estimator_)
], voting='soft')

ensemble.fit(x_train, y_train)
ensemble_preds = ensemble.predict(x_test)
models = {
    'Logistic Regression': (lr_preds, lr),
    'K-Nearest Neighbors': (knn_preds, knn.best_estimator_),
    'Random Forest': (rf_preds, rf.best_estimator_),
    'Gradient Boosting': (gb_preds, gb.best_estimator_),
    'Voting Classifier': (ensemble_preds, ensemble)
}
for model_name, (preds, model) in models.items():
    accuracy = accuracy_score(y_test, preds)
    print(f'{model_name} Accuracy: {accuracy}')
    print(classification_report(y_test, preds))


Logistic Regression Accuracy: 0.795260663507109
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1549
           1       0.64      0.53      0.58       561

    accuracy                           0.80      2110
   macro avg       0.74      0.71      0.72      2110
weighted avg       0.79      0.80      0.79      2110

K-Nearest Neighbors Accuracy: 0.7744075829383886
              precision    recall  f1-score   support

           0       0.82      0.88      0.85      1549
           1       0.59      0.48      0.53       561

    accuracy                           0.77      2110
   macro avg       0.71      0.68      0.69      2110
weighted avg       0.76      0.77      0.77      2110

Random Forest Accuracy: 0.7971563981042654
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1549
           1       0.66      0.49      0.56       561

    accuracy                           0.8